In [ ]:
    ############    #############   Retries and timeouts   #############   ##############   

 =>  Phase: P0 -- FDE Foundations & Engineering Baseline
 =>  Topic: 0.1 Production Python
 =>  Week:  Week 1
 =>  Track: Core

 =>  Sections to fill in:
       1. Theory
       2. Diagram(s) / flowchart(s) (images/ folder)  -- only where the concept needs one
       3. Command / config reference (if applicable)
       4. Runnable code demo(s)
       5. Hands-on lab checklist
       6. Common pitfalls / notes


In [ ]:
    ############    #############   Retries and Timeouts   #############   ##############   

 =>  Only retry errors that are actually transient (timeouts, 429/503) -- retrying a 400
       (bad request) just repeats the same failure.

 =>  Exponential backoff (with jitter) spaces out retries so a struggling downstream service
       isn't hit with a synchronized retry storm.

 =>  Always pair retries with a deadline/timeout -- an unbounded retry loop can hang a
       request forever even though each individual attempt eventually fails fast.


<img src="images/retry-backoff-timeline.png" alt="Retry with exponential backoff: attempt 1 fail, attempt 2 fail, attempt 3 success">

In [ ]:
import asyncio
import random

class TransientError(Exception):
    pass

async def flaky_call(attempt_tracker: list) -> str:
    attempt_tracker.append(1)
    if len(attempt_tracker) < 3:
        raise TransientError("upstream temporarily unavailable")
    return "success"

async def call_with_retry(coro_fn, max_attempts: int = 5, base_delay: float = 0.2):
    for attempt in range(1, max_attempts + 1):
        try:
            return await asyncio.wait_for(coro_fn(), timeout=1.0)
        except (TransientError, asyncio.TimeoutError) as exc:
            if attempt == max_attempts:
                raise
            delay = base_delay * (2 ** (attempt - 1)) + random.uniform(0, 0.1)
            print(f"attempt {attempt} failed ({exc}), retrying in {delay:.2f}s")
            await asyncio.sleep(delay)

attempts: list = []
result = await call_with_retry(lambda: flaky_call(attempts))
print("final result:", result, "after", len(attempts), "attempts")


In [ ]:
 =>  The delay roughly doubles each retry (0.2s, 0.4s, 0.8s, ...) plus a small random
       jitter -- this is exponential backoff with jitter, the standard pattern for calling
       any external/LLM API.

 =>  asyncio.wait_for enforces a per-attempt deadline; combined with max_attempts, the whole
       call is bounded in total time no matter how the downstream behaves.


In [ ]:
    ############    #############   Hands-on Lab Checklist   #############   ##############   

 =>  [ ] Swap the hand-written retry loop for the 'tenacity' library and reproduce the same
           behaviour (exponential backoff, jitter, max attempts).

 =>  [ ] Add a 'retry only on these specific exception types' rule, and confirm a
           non-transient error (e.g. a 400-equivalent) is NOT retried.


In [ ]:
    ############    #############   Common Pitfalls   #############   ##############   

 =>  Retrying non-idempotent operations (e.g. 'create a payment') without an idempotency
       key -- a retried request can create a duplicate side effect.

 =>  No jitter -- if many clients retry on the exact same fixed schedule, they all hammer
       the downstream service again at the same moment (a retry storm).
